# Первичный аудит и предобработка данных (ДЗ 1)
**Кейс:** Исследование продаж и факторов, влияющих на динамику выручки в розничной торговле.  
**Цель ноутбука:** проверить качество исходных таблиц (`stores`, `features`, `sales`), выявить аномалии и расхождения с ТЗ, собрать единую витрину для дальнейшего EDA и проверки гипотез.

**План работы:**
1. Загрузка данных из локального хранилища / репозитория и проверка структуры.
2. Аудит качества: пропуски, дубликаты, форматы дат и выбросы.
3. Проверка ключей, объединение таблиц и генерация базовых признаков.
4. Фиксация выводов и вопросов к заказчику.

In [1]:
import pandas as pd
import numpy as np

stores = pd.read_csv("sample_data/stores_data_set.csv")
features = pd.read_csv("sample_data/Features_data_set.csv")
sales = pd.read_csv("sample_data/sales_data_set.csv")

### 1. Проверка типов данных и приведение дат
При первичном осмотре видно, что даты в `features` и `sales` записаны в европейском формате `DD/MM/YYYY` (например, `19/02/2010`). При стандартном парсинге без указания формата день и месяц могут перепутаться, поэтому приводим их явно через `format='%d/%m/%Y'`.

In [2]:
features["Date"] = pd.to_datetime(features["Date"], format="%d/%m/%Y")
sales["Date"] = pd.to_datetime(sales["Date"], format="%d/%m/%Y")

print(f"Период в sales:    {sales['Date'].min():%d.%m.%Y} — {sales['Date'].max():%d.%m.%Y} ({sales['Date'].nunique()} нед.)")
print(f"Период в features: {features['Date'].min():%d.%m.%Y} — {features['Date'].max():%d.%m.%Y} ({features['Date'].nunique()} нед.)")
print(f"Уникальных магазинов: {sales['Store'].nunique()}, отделов: {sales['Dept'].nunique()}")

Период в sales:    05.02.2010 — 26.10.2012 (143 нед.)
Период в features: 05.02.2010 — 26.07.2013 (182 нед.)
Уникальных магазинов: 45, отделов: 81


### 2. Аудит пропусков и дубликатов
Проверим долю пропущенных значений (`NA`) и наличие дублей по бизнес-ключам:
* для `stores` — `Store`;
* для `features` — `(Store, Date)`;
* для `sales` — `(Store, Dept, Date)` (уровень наблюдения: магазин — отдел — неделя).

In [3]:
# Считаем пропуски только там, где они есть (в features)
na_summary = pd.DataFrame({
    "Пропусков (шт.)": features.isna().sum(),
    "Доля (%)": (features.isna().mean() * 100).round(2)
})
print("Пропуски в таблице features:")
display(na_summary[na_summary["Пропусков (шт.)"] > 0])

print(f"Пропуски в stores: {stores.isna().sum().sum()} | в sales: {sales.isna().sum().sum()}")

# Проверка дубликатов
dups = pd.Series({
    "stores (по Store)": stores.duplicated(subset=["Store"]).sum(),
    "features (по Store + Date)": features.duplicated(subset=["Store", "Date"]).sum(),
    "sales (по Store + Dept + Date)": sales.duplicated(subset=["Store", "Dept", "Date"]).sum(),
})
print("\nДубликаты по ключам:")
display(dups.to_frame("Количество дублей"))

Пропуски в таблице features:


,Пропусков (шт.),Доля (%)
MarkDown1,4158,50.77
MarkDown2,5269,64.33
MarkDown3,4577,55.89
MarkDown4,4726,57.70
MarkDown5,4140,50.55
CPI,585,7.14
Unemployment,585,7.14


Пропуски в stores: 0 | в sales: 0

Дубликаты по ключам:


,Количество дублей
stores (по Store),0
features (по Store + Date),0
sales (по Store + Dept + Date),0


### 3. Проверка аномалий и выбросов
Посмотрим на описательные статистики числовых полей. Особое внимание уделим отрицательным значениям в недельных продажах (`Weekly_Sales`) и промо-уценках (`MarkDown1–5`).

In [4]:
display(sales[["Weekly_Sales"]].describe().T)

neg_sales = sales[sales["Weekly_Sales"] < 0]
zero_sales = sales[sales["Weekly_Sales"] == 0]
print(f"Отрицательные продажи: {len(neg_sales):,} строк ({len(neg_sales)/len(sales):.2%}), минимум: {sales['Weekly_Sales'].min():.2f}")
print(f"Нулевые продажи: {len(zero_sales)} строк")

# Проверка отрицательных значений в скидках
md_cols = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]
neg_md = pd.DataFrame({
    "Отрицательных значений": (features[md_cols] < 0).sum(),
    "Минимум": features[md_cols].min()
})
print("\nАномалии в промо-акциях (MarkDown):")
display(neg_md)

,count,mean,std,min,25%,50%,75%,max
Weekly_Sales,421570.0,15981.258123,22711.183519,-4988.94,2079.65,7612.03,20205.8525,693099.36


Отрицательные продажи: 1,285 строк (0.30%), минимум: -4988.94
Нулевые продажи: 73 строк

Аномалии в промо-акциях (MarkDown):


,Отрицательных значений,Минимум
MarkDown1,4,-2781.45
MarkDown2,25,-265.76
MarkDown3,13,-179.26
MarkDown4,0,0.22
MarkDown5,2,-185.17


### 4. Объединение таблиц и подготовка признаков (с учётом ответов заказчика)
Перед объединением проверим согласованность `IsHoliday` в `sales` и `features`. Присоединим справочники через `left join` к `sales` (отсекая период 2013 г. без продаж).

**Правила предобработки, согласованные с заказчиком:**
1. **Пропуски в `MarkDown1–5`:** данные собирались с ноября 2011 г. не для всех магазинов и недель, поэтому мы **не приравниваем автоматически `NA` к нулю**. Вместо этого создаём флаг доступности промо-данных (`Has_MarkDown_Data`) и считаем суммарный промо-бюджет только для наблюдений, где зафиксирован хотя бы один канал уценок (`skipna=True`).
2. **Отрицательные значения `MarkDown1–5`:** выделяем флагом `Has_Neg_MarkDown` для последующего анализа чувствительности при оценке промо-эффекта (исключение строк vs обнуление корректировок).
3. **Отрицательные `Weekly_Sales`:** подтверждены заказчиком как возвраты товаров и сохранены для расчёта чистой выручки.


In [5]:
# Проверка согласованности IsHoliday перед слиянием
holiday_check = (
    sales[["Store", "Date", "IsHoliday"]].drop_duplicates()
    .merge(features[["Store", "Date", "IsHoliday"]], on=["Store", "Date"], suffixes=("_s", "_f"))
)
assert (holiday_check["IsHoliday_s"] == holiday_check["IsHoliday_f"]).all(), "Есть расхождения в IsHoliday!"

# Сборка единой витрины
df = (
    sales
    .merge(features, on=["Store", "Date", "IsHoliday"], how="left")
    .merge(stores, on="Store", how="left")
)

# Календарные признаки
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Week"] = df["Date"].dt.isocalendar().week.astype(int)

# Работа с MarkDown1-5 по правилам заказчика (без слепого fillna(0))
df["Is_MarkDown_Period"] = (df["Date"] >= "2011-11-11").astype(int)
df["Has_MarkDown_Data"] = df[md_cols].notna().any(axis=1).astype(int)

# Флаг строк с отрицательными корректировками MarkDown (для анализа чувствительности)
df["Has_Neg_MarkDown"] = (df[md_cols] < 0).any(axis=1).astype(int)

# Суммарный объём уценок считаем только там, где есть данные по промо (иначе оставляем NaN)
df["Total_MarkDown"] = np.where(
    df["Has_MarkDown_Data"] == 1,
    df[md_cols].clip(lower=0).sum(axis=1, skipna=True),
    np.nan
)
df["Has_MarkDown"] = np.where(
    df["Has_MarkDown_Data"] == 1,
    (df["Total_MarkDown"] > 0).astype(int),
    np.nan
)

# Температура в градусах Цельсия
df["Temp_C"] = ((df["Temperature"] - 32) * 5 / 9).round(1)

print(f"Итоговая витрина: {df.shape[0]:,} строк, {df.shape[1]} столбцов")
print(f"Строк с доступными данными по MarkDown: {df['Has_MarkDown_Data'].sum():,} ({df['Has_MarkDown_Data'].mean():.1%})")
print(f"Строк с отрицательными MarkDown в периоде продаж: {df['Has_Neg_MarkDown'].sum():,}")
display(df.head(5))

Итоговая витрина: 421,570 строк, 25 столбцов
Строк с доступными данными по MarkDown: 151,432 (35.9%)
Строк с отрицательными MarkDown в периоде продаж: 1,568


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,...,Size,Year,Month,Week,Is_MarkDown_Period,Has_MarkDown_Data,Has_Neg_MarkDown,Total_MarkDown,Has_MarkDown,Temp_C
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,...,151315,2010,2,5,0,0,0,NaN,NaN,5.7
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,...,151315,2010,2,6,0,0,0,NaN,NaN,3.6
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,...,151315,2010,2,7,0,0,0,NaN,NaN,4.4
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,...,151315,2010,2,8,0,0,0,NaN,NaN,8.1
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,...,151315,2010,3,9,0,0,0,NaN,NaN,8.1


### 5. Итоги аудита и согласованные решения с заказчиком

**Результаты проверки и очистки:**
* **Временной охват:** `features` охватывает 182 недели (по 26.07.2013), `sales` — 143 недели (по 26.10.2012). При `left join` к продажам период без целевой переменной корректно отсечён вместе с пропусками в `CPI` и `Unemployment` (которые начинаются с мая 2013 г.).
* **Формат дат и ключи:** даты приведены из `DD/MM/YYYY` к `datetime`, дублей по ключам нет, признак `IsHoliday` полностью согласован (0 расхождений).

**Зафиксированные решения по итогам коммуникации с заказчиком:**
1. **Гранулярность и сегментация:** сегментация клиентов исключена (отсутствует `Customer ID`); вместо неё в пул приоритетных направлений включена поведенческая сегментация отделов и магазинов.
2. **Промо-данные (`MarkDown1–5`):** пропуски не приравниваются автоматически к отсутствию акции; создана маска `Has_MarkDown_Data` для корректного анализа промо-эффекта только на заполненных наблюдениях.
3. **Отрицательные значения:** отрицательные `Weekly_Sales` подтверждены как возвраты и оставлены в расчётах чистой выручки; для редких отрицательных `MarkDown` создан флаг `Has_Neg_MarkDown` для проверки устойчивости выводов (исключение строк vs обнуление корректировок).
4. **Регионы:** региональный разрез официально снят заказчиком ввиду отсутствия признака в `stores.csv`.
